# affect_aif Paper Reproduction Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/har5h1l/affect_aif/blob/master/notebooks/reproduce.ipynb)

Run the paper reproduction configs, regenerate compact analysis artifacts, plot paper readouts, and export a clean `results/` folder. This notebook is Colab-compatible and local-Jupyter-compatible.

Top-to-bottom execution runs experiments first into `outputs/paper_full_*`, materializes those fresh outputs into the numbered canonical gitignored `results/paper/*/raw/` layout, and then analyzes the materialized files one paper section at a time.


## What This Notebook Proves

This notebook is the full numbered paper route. It does not assume paper data is already present. Each experiment section includes:

- the TOML config path;
- a dry-run manifest for run counts;
- the real `scripts/experiment/run.py` execution cell;
- materialization into the canonical `results/paper/` layout;
- analysis and plotting cells for that experiment.

Use `notebooks/demo.ipynb` first if you only want a quick workflow check.


## 1. Bootstrap Runtime

In Colab this clones the repo into `/content/affect_aif`. Locally, launch Jupyter from the repo root. GPU is optional; use a GPU runtime if available.


In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap

IN_COLAB = Path("/content").exists()
REPO_URL = os.environ.get("AFFECT_AIF_REPO_URL", "https://github.com/har5h1l/affect_aif.git")
BRANCH = os.environ.get("AFFECT_AIF_BRANCH", "master")

if IN_COLAB:
    os.chdir("/content")
    if not Path("affect_aif").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, "affect_aif"], check=True)
    os.chdir("/content/affect_aif")

ROOT = Path.cwd()
if not (ROOT / "scripts" / "experiment" / "run.py").exists():
    raise RuntimeError("Run this notebook from the affect_aif repo root, or use the Colab bootstrap clone.")

print("Repo root:", ROOT)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())


## 2. Install Dependencies

Colab runtimes start clean, so install the project into the runtime. Local users with an existing editable install can set `INSTALL_DEPS = False`.


In [ ]:
INSTALL_DEPS = True

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
    print("Installed affect_aif in editable mode for this runtime.")
else:
    print("Skipped dependency install.")


## 3. Check Runtime Device

This reports whether JAX sees an accelerator. The experiments should run on CPU too; the GPU check is for transparency and Colab readiness.


In [ ]:
try:
    import jax
    print("JAX devices:", jax.devices())
except Exception as exc:
    print("JAX device check unavailable:", exc)

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No NVIDIA GPU visible. This notebook also runs on CPU; GPU is optional for these trust-task sweeps.")


## 4. Paper Controls And Helpers

`RUN_EXPERIMENTS = True` means every experiment section executes its config before analysis. `MATERIALIZE_RESULTS = True` replaces the canonical raw folder for that experiment with the fresh notebook output so later analysis cannot silently use stale local data.


In [ ]:
RUN_EXPERIMENTS = True
MATERIALIZE_RESULTS = True
RUN_ANALYSIS = True
EXPORT_TO_DRIVE = False
RESET_OUTPUTS = False

OUTPUT_ROOT = Path("outputs")
PAPER_BATCH_PREFIX = "paper_full"
MANUSCRIPT_DIR = Path("docs/manuscript")
RESULTS_ROOT = Path("results")

PAPER_CONFIGS = {
    "predictability_value": "configs/paper/01_predictability_value.toml",
    "deployment_ablation": "configs/paper/02_deployment_ablation.toml",
    "partner_selection": "configs/paper/03_partner_selection.toml",
    "betrayal_adaptation": "configs/paper/04_betrayal_adaptation.toml",
    "alpha_sweep": "configs/paper/05a_alpha_sweep.toml",
    "prior_factorial": "configs/paper/05b_prior_factorial.toml",
    "forgiveness": "configs/paper/05c_forgiveness.toml",
}

for name, config in PAPER_CONFIGS.items():
    if not Path(config).exists():
        raise FileNotFoundError(f"Missing paper config for {name}: {config}")

CANONICAL_RAW = {
    "predictability_value": Path("results/paper/01_predictability_value/raw/predictability_value/predictability_value/results.csv"),
    "deployment_ablation": Path("results/paper/02_deployment_ablation/raw/deployment_ablation/deployment_ablation/results.csv"),
    "partner_selection": Path("results/paper/03_partner_selection/raw/partner_selection/partner_selection/results.csv"),
    "betrayal_adaptation": Path("results/paper/04_betrayal_adaptation/raw/betrayal_adaptation/betrayal_adaptation/results.csv"),
    "alpha_sweep": Path("results/paper/05a_alpha_sweep/raw/results.csv"),
    "prior_factorial": Path("results/paper/05b_prior_factorial/raw/results.csv"),
    "forgiveness": Path("results/paper/05c_forgiveness/raw/results.csv"),
}

import pandas as pd
import matplotlib.pyplot as plt


def run_required(cmd, required_path: Path | None = None):
    if required_path is not None and not required_path.exists():
        raise FileNotFoundError(f"Missing required file: {required_path}")
    print("Running:", " ".join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], check=True)


def batch_name(name: str) -> str:
    return f"{PAPER_BATCH_PREFIX}_{name}"


def batch_dir(name: str) -> Path:
    return OUTPUT_ROOT / batch_name(name)


def dry_run_config(name: str):
    dry_batch = batch_name(name) + "_dry"
    cmd = [
        sys.executable,
        "scripts/experiment/run.py",
        "--config",
        PAPER_CONFIGS[name],
        "--output-dir",
        str(OUTPUT_ROOT),
        "--batch-name",
        dry_batch,
        "--workers",
        "1",
        "--dry-run",
    ]
    run_required(cmd)
    manifest_path = OUTPUT_ROOT / dry_batch / "manifest.json"
    manifest = json.loads(manifest_path.read_text())
    frame = pd.DataFrame(manifest["configs"])
    display(frame[["hypothesis_id", "experiment_id", "rounds", "replications", "expanded_runs", "path"]])
    print("Total expanded runs:", int(frame["expanded_runs"].sum()))
    return frame


def run_paper_config(name: str):
    out = batch_dir(name)
    if RESET_OUTPUTS and out.exists():
        shutil.rmtree(out)
    cmd = [
        sys.executable,
        "scripts/experiment/run.py",
        "--config",
        PAPER_CONFIGS[name],
        "--output-dir",
        str(OUTPUT_ROOT),
        "--batch-name",
        batch_name(name),
        "--workers",
        "1",
    ]
    if RUN_EXPERIMENTS:
        run_required(cmd)
    else:
        print("Experiment run skipped. Set RUN_EXPERIMENTS = True to execute", name)
    if not out.exists():
        raise FileNotFoundError(f"Missing batch output for {name}: {out}")
    return out


def replace_dir(src: Path, dst: Path):
    if not src.exists():
        raise FileNotFoundError(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)


def combine_results(paths, output_path: Path):
    if not paths:
        raise RuntimeError(f"No child results available for {output_path}")
    frames = [pd.read_csv(path, low_memory=False) for path in paths]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    pd.concat(frames, ignore_index=True).to_csv(output_path, index=False)


def materialize_single(name: str, src: Path, dst: Path):
    if MATERIALIZE_RESULTS:
        replace_dir(src, dst)
    if not (dst / "results.csv").exists():
        raise FileNotFoundError(dst / "results.csv")
    return dst / "results.csv"


def materialize_suite(name: str, src_root: Path, dst_root: Path):
    if MATERIALIZE_RESULTS:
        if not src_root.exists():
            raise FileNotFoundError(src_root)
        dst_root.mkdir(parents=True, exist_ok=True)
        child_results = []
        for child in sorted(src_root.iterdir()):
            if not child.is_dir():
                continue
            dst = dst_root / child.name
            replace_dir(child, dst)
            result_path = dst / "results.csv"
            if result_path.exists():
                child_results.append(result_path)
        combine_results(child_results, dst_root / "results.csv")
    if not (dst_root / "results.csv").exists():
        raise FileNotFoundError(dst_root / "results.csv")
    return dst_root / "results.csv"


def analyze_core(name: str):
    raw = CANONICAL_RAW[name]
    if RUN_ANALYSIS:
        run_required(
            [
                sys.executable,
                "scripts/analysis/analyze.py",
                "--results",
                raw,
                "--output-dir",
                raw.parent,
                "--config",
                PAPER_CONFIGS[name],
            ],
            required_path=raw,
        )
    else:
        print("Analysis skipped. Set RUN_ANALYSIS = True to analyze", name)


def analyze_phenotype(name: str):
    raw = CANONICAL_RAW[name]
    if RUN_ANALYSIS:
        run_required(
            [
                sys.executable,
                "scripts/analysis/phenotype_artifacts.py",
                name,
                "--results-root",
                raw.parent,
                "--paper-dir",
                MANUSCRIPT_DIR,
            ],
            required_path=raw,
        )
    else:
        print("Analysis skipped. Set RUN_ANALYSIS = True to analyze", name)


def load_csv(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def display_metric_bar(table, title, metric, by="variant_id"):
    if table.empty or metric not in table.columns or by not in table.columns:
        print(f"Cannot plot {title}: missing {metric} or {by}.")
        return
    plot = table.groupby(by, as_index=False)[metric].mean()
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.bar(plot[by].astype(str), plot[metric])
    ax.set(title=title, xlabel=by, ylabel=metric)
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    plt.show()


def best_variant(table, metric, higher=True):
    if table.empty or metric not in table.columns or "variant_id" not in table.columns:
        return "unavailable"
    grouped = table.groupby("variant_id", as_index=False)[metric].mean().dropna()
    if grouped.empty:
        return "unavailable"
    row = grouped.sort_values(metric, ascending=not higher).iloc[0]
    return f"{row.variant_id} ({metric}={row[metric]:.3g})"


## 5. Predictability-Value: Run And Analyze

Config: `configs/paper/01_predictability_value.toml`

This section reproduces the predictability-over-value readout used in Section 3.1. It materializes fresh output under `results/paper/01_predictability_value/raw/predictability_value/predictability_value/`.


In [ ]:
pv_manifest = dry_run_config("predictability_value")
pv_batch = run_paper_config("predictability_value")
pv_raw = materialize_single(
    "predictability_value",
    pv_batch / "predictability_value" / "predictability_value",
    CANONICAL_RAW["predictability_value"].parent,
)
analyze_core("predictability_value")
print("Predictability-value raw result:", pv_raw)


## 6. Deployment Ablation: Run And Analyze

Config: `configs/paper/02_deployment_ablation.toml`

This section reproduces the tracked-only deployment ablation used in Section 3.2.


In [ ]:
deployment_manifest = dry_run_config("deployment_ablation")
deployment_batch = run_paper_config("deployment_ablation")
deployment_raw = materialize_single(
    "deployment_ablation",
    deployment_batch / "deployment_ablation" / "deployment_ablation",
    CANONICAL_RAW["deployment_ablation"].parent,
)
analyze_core("deployment_ablation")
print("Deployment-ablation raw result:", deployment_raw)


## 7. Partner Selection: Run And Analyze

Config: `configs/paper/03_partner_selection.toml`

This section reproduces the partner-selection sharpening readout used in Section 3.3.


In [ ]:
selection_manifest = dry_run_config("partner_selection")
selection_batch = run_paper_config("partner_selection")
selection_raw = materialize_single(
    "partner_selection",
    selection_batch / "partner_selection" / "partner_selection",
    CANONICAL_RAW["partner_selection"].parent,
)
analyze_core("partner_selection")
print("Partner-selection raw result:", selection_raw)


## 8. Betrayal Adaptation: Run And Analyze

Config: `configs/paper/04_betrayal_adaptation.toml`

This is the paper betrayal-adaptation confirmation run. It materializes fresh output under `results/paper/04_betrayal_adaptation/raw/betrayal_adaptation/betrayal_adaptation/`.


In [ ]:
h5_manifest = dry_run_config("betrayal_adaptation")
h5_batch = run_paper_config("betrayal_adaptation")
h5_raw = materialize_single(
    "betrayal_adaptation",
    h5_batch / "betrayal_adaptation" / "betrayal_adaptation",
    CANONICAL_RAW["betrayal_adaptation"].parent,
)
analyze_core("betrayal_adaptation")
print("Betrayal-adaptation raw result:", h5_raw)


### H5 Betrayal-Adaptation Readout

This plots a generated H5 summary table when available and shows the first rows for inspection.


In [ ]:
h5_phase_path = CANONICAL_RAW["betrayal_adaptation"].parent / "analysis" / "raw" / "betrayal_phase_summary.csv"
h5_final_path = CANONICAL_RAW["betrayal_adaptation"].parent / "analysis" / "raw" / "final_round_summary.csv"
h5_table = load_csv(h5_phase_path) if h5_phase_path.exists() else load_csv(h5_final_path)
display(h5_table.head())
for candidate in ["q_pi_entropy", "total_payoff", "payoff"]:
    if candidate in h5_table.columns:
        display_metric_bar(h5_table, f"H5 betrayal adaptation: {candidate}", candidate)
        break
print("Betrayal readout: compare entropy, accuracy, and payoff artifacts before making stronger behavioral claims.")


## 9. Exp A Alpha Sweep: Run And Analyze

Config: `configs/paper/05a_alpha_sweep.toml`

This config expands into two paper sub-experiments, `open_graded` and `betrayal`. The section runs both, materializes both child folders under `results/paper/05a_alpha_sweep/raw/`, combines their fresh `results.csv` files, and regenerates the compact phenotype metrics and figure.


In [ ]:
alpha_manifest = dry_run_config("alpha_sweep")
alpha_batch = run_paper_config("alpha_sweep")
alpha_raw = materialize_suite("alpha_sweep", alpha_batch / "exp_a", CANONICAL_RAW["alpha_sweep"].parent)
analyze_phenotype("alpha_sweep")
print("Exp A raw result:", alpha_raw)


### Exp A Alpha-Sweep Readout

This plots late entropy from the regenerated compact `metrics.csv`.


In [ ]:
alpha_metrics = load_csv("results/paper/05a_alpha_sweep/raw/metrics.csv")
display(alpha_metrics.head())
display_metric_bar(alpha_metrics, "Exp A: alpha sweep late entropy", "entropy_late")
print("Exp A readout: lowest late entropy:", best_variant(alpha_metrics, "entropy_late", higher=False))


## 10. Exp B Prior Factorial: Run And Analyze

Config: `configs/paper/05b_prior_factorial.toml`

This config expands into `open_graded`, `betrayal`, and `partner_choice`. The section materializes all children under `results/paper/05b_prior_factorial/raw/` and regenerates compact phenotype artifacts.


In [ ]:
prior_manifest = dry_run_config("prior_factorial")
prior_batch = run_paper_config("prior_factorial")
prior_raw = materialize_suite("prior_factorial", prior_batch / "exp_b", CANONICAL_RAW["prior_factorial"].parent)
analyze_phenotype("prior_factorial")
print("Exp B raw result:", prior_raw)


### Exp B Prior-Factorial Readout

This plots trust asymmetry from the regenerated compact `metrics.csv`.


In [ ]:
prior_metrics = load_csv("results/paper/05b_prior_factorial/raw/metrics.csv")
display(prior_metrics.head())
display_metric_bar(prior_metrics, "Exp B: prior factorial trust asymmetry", "trust_asymmetry")
print("Exp B readout: strongest trust asymmetry:", best_variant(prior_metrics, "trust_asymmetry", higher=True))


## 11. Exp C Forgiveness: Run And Analyze

Config: `configs/paper/05c_forgiveness.toml`

This section runs the trust-repair/reengagement experiment, materializes the raw folder, and regenerates the forgiveness metrics and figure.


In [ ]:
forgiveness_manifest = dry_run_config("forgiveness")
forgiveness_batch = run_paper_config("forgiveness")
forgiveness_raw = materialize_single(
    "forgiveness",
    forgiveness_batch / "exp_c" / "forgiveness",
    CANONICAL_RAW["forgiveness"].parent,
)
analyze_phenotype("forgiveness")
print("Exp C raw result:", forgiveness_raw)


### Exp C Forgiveness Readout

This plots reengagement from the regenerated compact `metrics.csv`.


In [ ]:
forgiveness_metrics = load_csv("results/paper/05c_forgiveness/raw/metrics.csv")
display(forgiveness_metrics.head())
display_metric_bar(forgiveness_metrics, "Exp C: reengagement rate", "reengagement_rate")
print("Exp C readout: strongest reengagement:", best_variant(forgiveness_metrics, "reengagement_rate", higher=True))


## 11. Verify Result Packet Cleanliness

The local `results/` folder should be safe to copy to Drive: no checkpoints, no partial CSVs, raw files under semantic paper folders, and compact summaries beside each result folder.


In [ ]:
partial_files = sorted(Path("results").glob("**/results_partial.csv")) + sorted(Path("results").glob("**/checkpoint_manifest.json"))
if partial_files:
    raise RuntimeError("Partial/checkpoint files remain:\n" + "\n".join(map(str, partial_files[:20])))

raw_count = sum(1 for _ in Path("results/paper").glob("**/raw/**/results.csv")) if Path("results/paper").exists() else 0
print("Paper raw results.csv count:", raw_count)

for name, path in CANONICAL_RAW.items():
    print(f"{name:22s}", "OK" if path.exists() else "missing", path)

if shutil.which("git"):
    probe = Path("results/paper/05a_alpha_sweep/raw/results.csv")
    if probe.exists():
        check = subprocess.run(["git", "check-ignore", "-q", str(probe)], check=False)
        print("Raw result gitignored:", check.returncode == 0)


## 12. Final Interpretation Scaffold

This cell prints a compact, table-derived interpretation scaffold from the experiment sections above. Use it as a reproducibility readout, not as a substitute for the manuscript text.


## 13. Export Clean Results To Google Drive

After full reproduction and analysis, this copies the local `results/` folder to Drive. The folder is designed to be shareable: compact summaries are tracked; raw paper data lives under `results/paper/*/raw`; checkpoints and partial CSVs should be absent.


In [ ]:
if EXPORT_TO_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Google Drive export is only available in Colab.")
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dest = Path("/content/drive/MyDrive/affect_aif_results")
    drive_dest.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-a", "results/", str(drive_dest) + "/"], check=True)
    print("Copied results/ to", drive_dest)
else:
    print("Drive export skipped. Set EXPORT_TO_DRIVE = True to copy results/ to Google Drive in Colab.")
